# Wear Classification Pipeline — Kaggle

**TFM — Máster en IA, UNIR**

Runs 2 classification branches on threading tap images with LOO-CV evaluation and MLflow tracking.

| Branch | Method | GPU | Approx. time (24 folds) |
|--------|--------|-----|-------------------------|
| A | MIL + EfficientNet-B0 | ✅ Required | ~90 min |
| B | U-Net + synthetic ISO masks | ✅ Required | ~60 min |

**Settings → Accelerator → GPU P100 before starting.**

## 1. Clone repo & install dependencies

In [2]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

# Add token in: Kaggle → Add-ons → Secrets → name: GITHUB_TOKEN
GITHUB_TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
REPO_USER    = "jorpeicas"  # ← cambia esto
REPO_NAME    = "TFM"
REPO_URL     = f"https://{GITHUB_TOKEN}@github.com/{REPO_USER}/{REPO_NAME}.git"
REPO_DIR     = "/kaggle/working/TFM"

if not os.path.exists(REPO_DIR):
    result = subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        err = result.stderr.replace(GITHUB_TOKEN, '***')
        raise RuntimeError(f"git clone failed:\n{err}")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")


Working dir: /kaggle/working/TFM


In [ ]:
import subprocess, sys

# Skip torch/torchvision — use Kaggle's pre-installed version (P100-compatible)
extra_deps = [
    "numpy==2.0.2",  # pin: satisfies Kaggle stack + our >=1.26 usage
    "opencv-python", "scipy", "pywavelets", "scikit-image",
    "scikit-learn", "matplotlib", "seaborn",
    "mlflow>=3.12.0", "timm>=1.0.26",
    "segmentation-models-pytorch>=0.5.0",
    "segment-anything>=1.0",
    "streamlit>=1.57.0",
    "anomalib",
    "grad-cam>=1.5.5",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + extra_deps,
    check=True
)

# Make src/ importable
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Link data from Kaggle Dataset

In [16]:
import sys

# ← Set this to your Kaggle dataset slug
# kaggle.com/datasets/TU_USUARIO/tfm-desgaste-data  →  slug = tfm-desgaste-data
KAGGLE_DATA = "/kaggle/input/datasets/jordiunir/dataset-tfm-roscas/data"  # ← cambia esto

# Symlink so pipeline finds data at ./data
if not os.path.exists("data"):
    os.symlink(KAGGLE_DATA, "data")

os.system("ls data/normal | head -5")
os.system("ls data/worn | head -5")

Imagen_000501_RB01.jpg
Imagen_000501_RB02.jpg
Imagen_000501_RB03.jpg
Imagen_000501_RB04.jpg
Imagen_000501_RB05.jpg
Imagen_000501_RM01.jpg
Imagen_000501_RM02.jpg
Imagen_000501_RM03.jpg
Imagen_000501_RM04.jpg
Imagen_000501_RM05.jpg


0

## 3. Configure MLflow (SQLite backend)

In [17]:
import yaml
from pathlib import Path

MLFLOW_URI = "sqlite:///mlflow.db"

# Patch all configs to use SQLite + GPU-friendly settings
for config_file in Path("configs").glob("*.yaml"):
    cfg = yaml.safe_load(config_file.read_text())
    cfg["mlflow_uri"] = MLFLOW_URI
    cfg["data_dir"] = "data"
    cfg["model_dir"] = "models"
    config_file.write_text(yaml.dump(cfg))
    print(f"Updated: {config_file.name}")

print(f"MLflow URI: {MLFLOW_URI}")

Updated: branch_a.yaml
Updated: branch_b.yaml
Updated: branch_e.yaml
Updated: branch_f.yaml
MLflow URI: sqlite:///mlflow.db


## Check progress (run after reconnecting)
If the session was interrupted, run this cell to see where each branch stopped.
Logs are saved in `/kaggle/working/` and persist across reconnections.

In [ ]:
# Run this cell after reconnecting to check branch progress
import os
LOG_DIR = "/kaggle/working"
for branch in ["A", "B", "C"]:
    log = f"{LOG_DIR}/branch_{branch.lower()}.log"
    if os.path.exists(log):
        lines = open(log).readlines()
        done = any("Summary run logged" in l or "complete" in l.lower() for l in lines)
        status = "DONE" if done else "running/interrupted"
        print(f"\n=== Branch {branch} [{status}] — {len(lines)} lines total ===")
        print("".join(lines[-20:]))
    else:
        print(f"\n=== Branch {branch}: not started ===")


## 4. Scale calibration (run once)

In [ ]:
import sys
sys.path.insert(0, "src")

from preprocessing.scale_calibration import calibrate_scale, save_scale_factor

SCALE_FILE = Path("scale_factor.json")

if SCALE_FILE.exists():
    from preprocessing.scale_calibration import load_scale_factor
    px_mm = load_scale_factor(SCALE_FILE)
    print(f"Loaded existing scale: {px_mm:.1f} px/mm")
else:
    # Use middle image from a normal tool for stable valley peaks
    normal_images = sorted(Path("data/normal").glob("Imagen_*"))
    ref_image = normal_images[len(normal_images) // 2]
    print(f"Calibrating from: {ref_image.name}")
    # valley_distance_mm=1.5 matches M10x1.5 pitch (valley-to-valley = 1 pitch)
    px_mm = calibrate_scale(ref_image, valley_distance_mm=1.5)
    save_scale_factor(px_mm, SCALE_FILE)
    print(f"Scale saved: {px_mm:.1f} px/mm")

In [ ]:
import subprocess, sys

LOG_DIR = "/kaggle/working"

def run_branch(branch: str, config: str) -> int:
    """Stream pipeline output to cell and persist to log file."""
    log_path = f"{LOG_DIR}/branch_{branch.lower()}.log"
    cmd = [sys.executable, "-u", "pipeline.py", "--branch", branch, "--config", config]
    print(f"\n▶ Branch {branch} — log: {log_path}\n", flush=True)
    with open(log_path, "w", buffering=1) as log:
        proc = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1
        )
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
        proc.wait()
    if proc.returncode != 0:
        print(f"\n✗ Branch {branch} FAILED (exit {proc.returncode}). Log: {log_path}")
    else:
        print(f"\n✓ Branch {branch} complete.")
    return proc.returncode


## 5. Run Branch A — MIL + EfficientNet-B0 (~90 min on T4)
**⏭ SKIP for this run — only Branch B (U-Net) is being executed.**
Requires GPU. Two-phase fine-tuning across 24 LOO-CV folds.


In [ ]:
# SKIPPED this run — only Branch B (U-Net) is being executed.
# run_branch("A", "configs/branch_a.yaml")


## 6. Run Branch B — U-Net + synthetic ISO masks (~60 min on T4)

In [ ]:
run_branch("B", "configs/branch_b.yaml")


## 7. Run Branch C — ISO Profile Comparison (no training, ~5 min)
**⏭ SKIP for this run — only Branch B (U-Net) is being executed.**


In [ ]:
# SKIPPED this run — only Branch B (U-Net) is being executed.
# run_branch("C", "configs/branch_c.yaml")


## 7. Quick results summary

In [ ]:
import mlflow
import pandas as pd

mlflow.set_tracking_uri(MLFLOW_URI)

rows = []
for branch in ["B"]:  # only Branch B this run
    try:
        runs = mlflow.search_runs(
            experiment_names=[f"branch_{branch}"],
            filter_string="tags.run_type = 'summary'",
        )
        if runs.empty:
            continue
        r = runs.sort_values("start_time", ascending=False).iloc[0]
        rows.append({
            "Branch": branch,
            "F1 mean": round(r.get("metrics.f1_mean", float("nan")), 3),
            "F1 std": round(r.get("metrics.f1_std", float("nan")), 3),
            "AUC-ROC mean": round(r.get("metrics.auc_roc_mean", float("nan")), 3),
            "AUPRC mean": round(r.get("metrics.auprc_mean", float("nan")), 3),
        })
    except Exception as e:
        print(f"Branch {branch}: {e}")

if rows:
    df = pd.DataFrame(rows).set_index("Branch")
    print(df.to_string())
else:
    print("No completed runs yet.")


## 8. Visualization pass (for Streamlit dashboard)
Train on all data once per branch and generate mask images for `output/vis/`.
Run this **after all LOO-CV cells complete**. Takes ~10 min total.

In [ ]:
for branch in ['B']:  # only Branch B this run
    print(f'\n=== Viz pass: Branch {branch} ===', flush=True)
    log_path = f"/kaggle/working/viz_{branch.lower()}.log"
    with open(log_path, 'w', buffering=1) as log:
        proc = subprocess.Popen(
            [sys.executable, '-u', 'pipeline.py',
             '--branch', branch,
             '--config', f'configs/branch_{branch.lower()}.yaml',
             '--viz'],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
        )
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line)
        proc.wait()
    if proc.returncode != 0:
        print(f'✗ Viz {branch} FAILED (exit {proc.returncode})')
    else:
        print(f'✓ Viz {branch} done.')

print('\nVisualizations saved to output/vis/')


## 9. Train final models
Trains each branch on **all 24 tools** (no LOO-CV). Saves a single deployable model per branch to `models/`.
Download these files alongside `mlflow.db` for local Streamlit inference.

In [ ]:
## 11. Train final model (Branch B only) — for local Streamlit inference
print('=== Training final Branch B model on full dataset ===', flush=True)
log_path = '/kaggle/working/train_final.log'
with open(log_path, 'w', buffering=1) as log:
    proc = subprocess.Popen(
        [sys.executable, '-u', 'train_final.py', '--branch', 'B'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
        log.write(line)
    proc.wait()
if proc.returncode != 0:
    print(f'✗ train_final.py FAILED (exit {proc.returncode})')
else:
    print('✓ Final model saved to models/')
    import os
    os.system('ls -lh models/')


## 10. Save results
Files in `/kaggle/working/` are preserved after the session ends.
Download from: Kaggle notebook → Output tab.

In [ ]:
import shutil, os, glob

# Zip everything Streamlit needs: MLflow DB + visualizations + raw console logs
# Output appears in Kaggle UI → Output tab → download
os.chdir('/kaggle/working/TFM')
os.makedirs('/kaggle/working/results', exist_ok=True)

# Copy files to a flat results/ dir so the zip is clean
if os.path.exists('mlflow.db'):
    shutil.copy('mlflow.db', '/kaggle/working/results/mlflow.db')
if os.path.exists('output/vis'):
    shutil.copytree('output/vis', '/kaggle/working/results/output/vis', dirs_exist_ok=True)

if os.path.exists('models'):
    shutil.copytree('models', '/kaggle/working/results/models', dirs_exist_ok=True)

# Raw console logs (per-tool tool_score lines) — needed to reconstruct per-tool
# correctness if a metric is ever missing from mlflow.db. Without these, a run
# is unrecoverable if mlflow logging has a gap (this happened before with Branch C).
os.makedirs('/kaggle/working/results/logs', exist_ok=True)
for log_file in glob.glob('/kaggle/working/*.log'):
    shutil.copy(log_file, '/kaggle/working/results/logs/')

shutil.make_archive('/kaggle/working/tfm_results', 'zip', '/kaggle/working/results')
print('Created: /kaggle/working/tfm_results.zip')
os.system('ls -lh /kaggle/working/tfm_results.zip')
print('\nContents:')
os.system('find /kaggle/working/results -type f')
print('\nDownload: Kaggle notebook → Output tab (right panel) → tfm_results.zip')

In [ ]:
# ── Local setup after downloading tfm_results.zip ──────────────────────────
# 1. Move zip to your local jordi/ folder:
#      mv ~/Downloads/tfm_results.zip /path/to/TFM-desgaste/jordi/
#
# 2. Extract (overwrites mlflow.db and output/):
#      cd TFM-desgaste/jordi && unzip -o tfm_results.zip
#
# 3. Launch dashboard:
#      uv run streamlit run app.py
#
# Streamlit reads:
#   - mlflow.db       → metrics (sections 1-4)
#   - output/vis/     → mask images (section 5)
#   - logs/           → raw per-tool console logs (backup, not read by Streamlit;
#                       keep for reconstructing per-tool scores / updating memoria)
#
# In the sidebar, select 'SQLite (./mlflow.db)' as backend.
print('See cell comments for local Streamlit setup.')